# Module 06 — Notebook 1: Script Structure

## Learning Objectives

By the end of this notebook you will be able to:
- Understand what `__name__` is and why `if __name__ == "__main__":` matters
- Structure a Python script with a `main()` function and a clean entry point
- Create a `.py` file from a notebook using `%%writefile`
- Import functions from your own modules
- Understand the difference between a script and a module

**Time:** ~20 minutes

In [ ]:
import sys
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_approx, check_contains
from pathlib import Path

# Create a scripts/ directory for this module's .py files
SCRIPTS_DIR = Path("scripts")
SCRIPTS_DIR.mkdir(exist_ok=True)
print("scripts/ directory ready")

## 1. `__name__` — The Most Important Python Variable You've Never Heard Of

Every Python file has a `__name__` variable. Its value depends on **how** the file is used:

| How the file is used | `__name__` value |
|---------------------|------------------|
| Run directly: `python my_script.py` | `"__main__"` |
| Imported: `import my_script` | `"my_script"` |

This is how Python distinguishes between "I am the program" and "I am a library being used".

In JavaScript, this distinction doesn't exist at the language level. Node.js has `require.main === module` for a similar check, but Python bakes this pattern in everywhere.

The pattern you'll see in every Python script:

```python
def main():
    print("Running!")

if __name__ == "__main__":
    main()
```

This means: **only call `main()` if this file is being run directly**. If another script imports this file, `main()` does not get called automatically — only the functions are made available.

In [ ]:
# In a notebook, __name__ is "__main__" (the notebook is the 'program')
print("Current __name__:", __name__)

# This is why the guard works in notebooks too
if __name__ == "__main__":
    print("This runs because we are in __main__ context")

## 2. Writing Python Files with `%%writefile`

`%%writefile filename.py` is a Jupyter cell magic that writes the cell's contents to a `.py` file.
This lets you develop scripts interactively in a notebook and then run them from the command line.

It's the primary tool you'll use in this module.

In [ ]:
%%writefile scripts/hello.py
# A minimal Python script

def greet(name):
    """Return a greeting string."""
    return f"Hello, {name}!"


def main():
    message = greet("AI Research Engineer")
    print(message)


if __name__ == "__main__":
    main()

In [ ]:
# Run the script from the notebook using !
!python scripts/hello.py

In [ ]:
# Import the module and use the function — main() does NOT run automatically
sys.path.insert(0, str(SCRIPTS_DIR))
import hello

result = hello.greet("pandas")
print(result)           # Hello, pandas!
print(hello.__name__)   # 'hello' — not '__main__'

## 3. The Standard Script Structure

Production Python scripts follow a consistent pattern:

```python
# 1. Imports at the top
import json
from pathlib import Path

# 2. Constants
THRESHOLD = 0.8

# 3. Helper functions (pure, testable, importable)
def load_data(path):
    with open(path) as f:
        return json.load(f)

def compute_flag_rate(outputs):
    flagged = sum(1 for o in outputs if o["flagged"])
    return flagged / len(outputs) if outputs else 0.0

# 4. main() — orchestrates the steps, uses the helpers
def main():
    outputs = load_data("data.json")
    rate = compute_flag_rate(outputs)
    print(f"Flag rate: {rate:.1%}")

# 5. Entry point guard
if __name__ == "__main__":
    main()
```

Why separate `main()` from the guard? Because it lets another script import and call `main()` with custom args later, without re-running all the side effects.

In JS this is like the difference between exporting a function (`module.exports = main`) and calling it (`main()`) — you can have both.

In [ ]:
%%writefile scripts/flag_summary.py
import json
from pathlib import Path

DATA_PATH = Path("../../data/synthetic/model_outputs.json")


def load_outputs(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def compute_flag_rate(outputs):
    if not outputs:
        return 0.0
    flagged = sum(1 for o in outputs if o["flagged"])
    return flagged / len(outputs)


def summarize(outputs):
    total     = len(outputs)
    flagged   = sum(1 for o in outputs if o["flagged"])
    flag_rate = flagged / total if total else 0.0
    return {"total": total, "flagged": flagged, "flag_rate": round(flag_rate, 4)}


def main():
    outputs = load_outputs(DATA_PATH)
    summary = summarize(outputs)
    print(f"Total outputs: {summary['total']}")
    print(f"Flagged:       {summary['flagged']}")
    print(f"Flag rate:     {summary['flag_rate']:.1%}")


if __name__ == "__main__":
    main()

In [ ]:
!python scripts/flag_summary.py

## 4. Importing From Your Own Module

Once a `.py` file exists, you can import its functions just like any library.
This is how you reuse logic across multiple scripts without copy-pasting.

In [ ]:
# Import the functions we just wrote
import importlib
import flag_summary
importlib.reload(flag_summary)   # reload in case it was already imported

# Call individual functions
test_outputs = [
    {"id": 1, "model": "model-a", "flagged": False},
    {"id": 2, "model": "model-b", "flagged": True},
    {"id": 3, "model": "model-b", "flagged": True},
]

rate    = flag_summary.compute_flag_rate(test_outputs)
summary = flag_summary.summarize(test_outputs)

print(f"Flag rate: {rate:.2f}")          # 0.67
print(f"Summary:   {summary}")           # {'total': 3, 'flagged': 2, 'flag_rate': 0.6667}

---
## Your Turn — Exercise 1: Write a Helper Function

Write a function `get_model_flag_rates(outputs)` that:
- Takes a list of output dicts (each has `"model"` and `"flagged"` keys)
- Returns a dict mapping each model name to its flag rate (as a float)

Test it on `sample_outputs` and store the result in `rates`.
Store the flag rate for `"model-b"` in `b_rate`, rounded to 2 decimal places.

In [ ]:
sample_outputs = [
    {"id": 1, "model": "model-a", "flagged": False},
    {"id": 2, "model": "model-a", "flagged": False},
    {"id": 3, "model": "model-b", "flagged": True},
    {"id": 4, "model": "model-b", "flagged": True},
    {"id": 5, "model": "model-b", "flagged": False},
]

# YOUR CODE HERE
def get_model_flag_rates(outputs):
    pass   # replace with your implementation

rates  = None   # call get_model_flag_rates(sample_outputs)
b_rate = None   # flag rate for "model-b", rounded to 2 decimal places

In [ ]:
check_type(rates, dict, "rates is a dict")
check_contains(rates, "model-a", "rates has model-a")
check_contains(rates, "model-b", "rates has model-b")
check_approx(rates["model-a"], 0.0, 1e-6, "model-a flag rate is 0.0")
check_approx(b_rate, 0.67, 0.01, "model-b flag rate is 0.67")

---
## Your Turn — Exercise 2: Write a Complete Script

Use `%%writefile` to create `scripts/model_rates.py` that:
1. Defines `get_model_flag_rates(outputs)` (your solution from Exercise 1)
2. Defines `main()` that loads `../../data/synthetic/model_outputs.json` and prints each model's flag rate
3. Has the `if __name__ == "__main__":` guard

Then run it. The script file should exist at `scripts/model_rates.py`.

In [ ]:
%%writefile scripts/model_rates.py
# YOUR CODE HERE — replace this comment with the full script
print("replace me")

In [ ]:
!python scripts/model_rates.py

In [ ]:
script_path = Path("scripts/model_rates.py")
check_equal(script_path.exists(), True, "scripts/model_rates.py exists")

# The script must contain the __name__ guard
source = script_path.read_text()
check_contains(source, "__name__", "script has __name__ guard")
check_contains(source, "def main", "script has a main() function")

---
## Why This Matters for AI Research Engineering

Notebooks are great for exploration, but research code that runs on a cluster, gets scheduled nightly, or is shared with teammates needs to live in `.py` files.

The `if __name__ == "__main__":` pattern is how you make scripts that are **both** runnable from the terminal **and** importable as libraries. This matters because:
- A teammate can `import flag_summary` to use your `compute_flag_rate()` in their script
- Your CI pipeline can `python run_evaluation.py --input data.json` without worrying about side effects
- You can test individual functions by importing them in a notebook — just like you did above

The `%%writefile` workflow is how you develop scripts in a notebook and then graduate them to standalone files — the typical path for research code that starts as exploration.

## Summary

| Concept | Code |
|---------|------|
| Check if running directly | `if __name__ == "__main__":` |
| Write a .py file | `%%writefile scripts/my_script.py` |
| Run a script | `!python scripts/my_script.py` |
| Import your module | `import my_module; my_module.my_func()` |
| Reload after edits | `importlib.reload(my_module)` |
| Standard structure | imports → constants → functions → `main()` → entry point |

**Next:** Notebook 2 — `argparse` and `logging` for production-quality scripts.